# Phase 2b — JetBot Sensor Server (with peer detection)

Same architecture as Phase 1b: this notebook runs **on the JetBot** and exposes the bot as a sensor + actuator over HTTP. New in 2b: each bot also detects other bots' identification charts (filled colored shapes printed on a vertical prism on each bot's chassis) and reports them in `/state`.

**This file is byte-identical across all 4 bots.** Per-bot identity (color + shape) is set at runtime by the client via `POST /assign_identity`. Until then, the bot reports all chart detections including its own (fail-open — Phase 2a workflows still work).

**Identification scheme:** 2 colors × 2 shapes = 4 unique bots. Yellow circle, yellow square, blue circle, blue square. Color is decided by HSV mask; shape by circularity.

**Endpoints (additions over Phase 1b in bold):**
- `GET  /state` — JSON with cube + frame_diff + **peers + assigned identity**
- `GET  /camera` — raw JPEG
- `GET  /camera_annotated` — JPEG with cube bbox **and peer bboxes**
- `GET  /health` — liveness + **assigned identity**
- `POST /command` — motors with TTL watchdog
- **`POST /assign_identity` — set this bot's color and shape**

**Pipeline per detection-thread iteration:**
1. Capture frame from camera
2. Convert BGR → HSV **once** (shared by cube and peer detection)
3. Run cube detection on the HSV image
4. For each known peer color, run chart detection on the HSV image; classify shape per detection
5. If identity is assigned, drop the peer matching this bot's own (color, shape)
6. Compute frame-difference value (for client's stuck detector)
7. Update `latest_sensor_state` under a lock

**Section structure:**
- §1.  Imports and hardware init
- §2.  Tunable parameters
- §3.  Identity assignment state
- §4.  Cube detection (refactored: takes HSV)
- §5.  Chart detection (color mask + shape classification + scoring + per-color)
- §6.  Combined detection (single BGR→HSV pass)
- §7.  Distance calibration (cube + per-(color, shape) chart)
- §8.  In-situ HSV recalibration (red + per-color chart)
- §9.  Background detection thread
- §10. HTTP server (with `/assign_identity`)
- §11. Start server
- §12. Detection-rate diagnostic
- §13. Shutdown


## §1. Imports and hardware initialization

In [ ]:
import cv2
import numpy as np
import time
import threading
import math
import json as json_module
from http.server import BaseHTTPRequestHandler, HTTPServer
import urllib.parse

from jetbot import Camera, Robot

# Same camera resolution as Phase 1a/1b so cube calibration carries over.
CAMERA_WIDTH_PIXELS  = 720
CAMERA_HEIGHT_PIXELS = 720

camera_instance = Camera.instance(width=CAMERA_WIDTH_PIXELS, height=CAMERA_HEIGHT_PIXELS)
robot_instance  = Robot()

print("Camera and robot initialized.")
print(f"Frame shape: {camera_instance.value.shape}")


## §2. Tunable parameters

Anything that affects what the bot **senses** lives here. Anything about what the bot **does** lives on the client.

This file is identical across bots — per-bot identity (color, shape) is set at runtime by the client via `POST /assign_identity`, not via edits to this notebook.

### §2.1 Cube detection parameters (carried over from Phase 1b)

In [ ]:
# --- Red HSV thresholds (red wraps around 0/180 so two ranges) ---
red_hue_lower_range_min = np.array([  0, 100,  60])
red_hue_lower_range_max = np.array([ 12, 255, 255])
red_hue_upper_range_min = np.array([165, 100,  60])
red_hue_upper_range_max = np.array([180, 255, 255])

# --- Cube acceptance filters ---
minimum_blob_area_pixels       = 80
minimum_aspect_ratio_for_cube  = 0.35
maximum_aspect_ratio_for_cube  = 2.80
minimum_solidity_for_cube      = 0.55

# --- Cube scoring weights ---
scoring_weight_for_area       = 0.5
scoring_weight_for_solidity   = 0.3
scoring_weight_for_squareness = 0.2

# --- Cube morphology ---
morphology_open_kernel_size   = 3
morphology_close_kernel_size  = 5

# --- Cube distance calibration ---
# Calibrated value from Phase 1a/1b at 720x720. Re-run §7.1 if anything changes.
distance_calibration_constant = 81.0

print("Cube detection parameters loaded.")


### §2.2 Chart detection parameters (NEW)

**Why these defaults:**
- Yellow centred at hue ~28; blue centred at hue ~113. Both >25 hue units from red (0–12, 165–180), >25 from chassis green (~60), and >70 from each other.
- Solidity floor 0.85 (vs cube's 0.55) — charts are flat filled rectangles; a real chart should be near-convex even partly occluded. Tighter floor rejects more spurious blobs (e.g. a yellow stripe on someone's bag).
- Aspect ratio acceptance 0.50–2.00 — generous because a chart on a prism face viewed from 60° off-axis foreshortens to ~0.5.
- Aspect ratio for **shape classification** is tighter (0.77–1.30): if a chart is too foreshortened to read its aspect cleanly, we report `peer_shape='unknown'` instead of guessing. A foreshortened circle and a foreshortened square both have circularity around 0.85 — neither classification is reliable.
- Circularity thresholds 0.83/0.88 leave a 'unknown' band of width 0.05 between square (~0.785 theoretical) and circle (~1.0 theoretical). In practice both shift down ~5-10% from theoretical due to pixel discretization — calibrate if classifications look wrong.

Run §8.2 to recalibrate any color's HSV range on hardware in your lighting.

In [ ]:
# --- HSV ranges per peer color. Calibrate in §8.2 if needed. ---
chart_color_hsv_ranges = {
    'yellow': (
        np.array([ 18, 100,  80], dtype=np.uint8),  # H_lo, S_lo, V_lo
        np.array([ 38, 255, 255], dtype=np.uint8),  # H_hi, S_hi, V_hi
    ),
    'blue': (
        np.array([ 98, 100,  60], dtype=np.uint8),
        np.array([128, 255, 255], dtype=np.uint8),
    ),
}

# --- Chart acceptance filters (tighter than cube) ---
minimum_chart_blob_area_pixels  = 80
minimum_chart_aspect_ratio      = 0.50   # accept foreshortened
maximum_chart_aspect_ratio      = 2.00
minimum_chart_solidity          = 0.85   # filled rectangles are convex

# --- Shape classification thresholds (only applied to accepted charts) ---
# If aspect is more than this much off 1:1, mark shape 'unknown' (foreshortening
# makes circle and square indistinguishable).
maximum_aspect_log_ratio_for_shape_classification = math.log(1.30)  # i.e. aspect in [0.77, 1.30]
circularity_threshold_for_circle = 0.88   # >= this -> circle
circularity_threshold_for_square = 0.83   # <= this -> square
                                           # in between -> 'unknown'

# --- Chart scoring weights ---
chart_scoring_weight_for_area              = 0.4
chart_scoring_weight_for_solidity          = 0.4
chart_scoring_weight_for_aspect_closeness  = 0.2

# --- Chart morphology (smaller than cube — flat charts have less noise) ---
chart_morphology_open_kernel_size  = 3
chart_morphology_close_kernel_size = 3

print("Chart detection parameters loaded.")


### §2.3 Per-(color, shape) chart distance calibration

`k_chart` depends on *both* color and shape because:
- HSV mask edge erosion is color-dependent (blue desaturates faster than yellow as light dims, so blue's measured pixel area is biased small at the same physical distance — over-estimates distance with single-k)
- Circle and square at the same nominal physical size have different filled areas (πr² vs s²) — unless you constrain the chart printer to make them equal-area, k differs

Calibrate each entry in §7.2.

In [ ]:
# Placeholder values; calibrate in §7.2 on hardware.
chart_distance_calibration_constants = {
    ('yellow', 'circle'): 162.0,
    ('yellow', 'square'): 162.0,
    ('blue',   'circle'): 162.0,
    ('blue',   'square'): 162.0,
}


def get_chart_distance_constant(color_name, shape_name):
    """Return k for distance estimation. For shape='unknown', use the average
    over known shapes for that color (best we can do without shape info)."""
    if shape_name == 'unknown':
        constants_for_color = [
            value for (c_key, s_key), value in chart_distance_calibration_constants.items()
            if c_key == color_name
        ]
        if not constants_for_color:
            return 0.0
        return sum(constants_for_color) / len(constants_for_color)
    return chart_distance_calibration_constants.get((color_name, shape_name), 0.0)


print("Per-(color, shape) k_chart loaded.")


### §2.4 Detection-loop and HTTP parameters

In [ ]:
# How often the background detection thread refreshes latest_sensor_state.
# 10 Hz matches Phase 1b. The §12 diagnostic measures whether we actually hit this.
detection_loop_period_seconds = 0.10

# HTTP server port (matches Phase 1b client expectations).
http_server_port = 8080

# Default command watchdog TTL when client doesn't pass one.
default_command_ttl_milliseconds = 500

# Valid identity sets — used by /assign_identity validation.
valid_chart_colors = set(chart_color_hsv_ranges.keys())  # {'yellow', 'blue'}
valid_chart_shapes = {'circle', 'square'}

print("Loop and HTTP parameters loaded.")
print(f"Valid chart colors: {sorted(valid_chart_colors)}")
print(f"Valid chart shapes: {sorted(valid_chart_shapes)}")


## §3. Identity assignment state

Module-level variables holding this bot's assigned identity. Set at runtime by `POST /assign_identity`. Defaults to `(None, None)` — fail-open: the detection thread reports all peer detections including this bot's own chart until identity is assigned.

In [ ]:
identity_lock = threading.Lock()
assigned_color = None  # 'yellow' | 'blue' | None
assigned_shape = None  # 'circle' | 'square' | None


def set_assigned_identity(new_color, new_shape):
    global assigned_color, assigned_shape
    with identity_lock:
        assigned_color = new_color
        assigned_shape = new_shape
    print(f"Identity assigned: color={new_color!r}, shape={new_shape!r}")


def get_assigned_identity():
    with identity_lock:
        return assigned_color, assigned_shape


print("Identity state initialized (assigned_color=None, assigned_shape=None).")


## §4. Cube detection (refactored to take HSV)

Logic is identical to Phase 1b, but `build_red_color_mask` and `detect_red_cube` now operate on a pre-converted HSV image rather than doing the conversion themselves. This is what enables the single BGR→HSV pass in §6, where one conversion is shared between cube detection and all peer-color detections.

The standalone `detect_red_cube(captured_frame_bgr)` wrapper is preserved for the smoke test and for backward-compatible debugging.

In [ ]:
def build_red_color_mask_from_hsv(frame_in_hsv):
    """Return a binary mask: 255 where red, 0 elsewhere. Caller does cvtColor."""
    mask_for_lower_red = cv2.inRange(frame_in_hsv, red_hue_lower_range_min, red_hue_lower_range_max)
    mask_for_upper_red = cv2.inRange(frame_in_hsv, red_hue_upper_range_min, red_hue_upper_range_max)
    combined_red_mask  = cv2.bitwise_or(mask_for_lower_red, mask_for_upper_red)

    open_kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                             (morphology_open_kernel_size, morphology_open_kernel_size))
    close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                             (morphology_close_kernel_size, morphology_close_kernel_size))
    cleaned_red_mask = cv2.morphologyEx(combined_red_mask, cv2.MORPH_OPEN,  open_kernel)
    cleaned_red_mask = cv2.morphologyEx(cleaned_red_mask,  cv2.MORPH_CLOSE, close_kernel)
    return cleaned_red_mask


def score_cube_candidate_contour(candidate_contour):
    candidate_area_pixels = float(cv2.contourArea(candidate_contour))
    bounding_box_x, bounding_box_y, bounding_box_w, bounding_box_h = cv2.boundingRect(candidate_contour)
    bounding_box_aspect_ratio = bounding_box_w / max(bounding_box_h, 1)

    convex_hull = cv2.convexHull(candidate_contour)
    convex_hull_area = float(cv2.contourArea(convex_hull))
    candidate_solidity = candidate_area_pixels / max(convex_hull_area, 1.0)

    candidate_properties = {
        'area':         candidate_area_pixels,
        'aspect_ratio': bounding_box_aspect_ratio,
        'solidity':     candidate_solidity,
        'bounding_box': (bounding_box_x, bounding_box_y, bounding_box_w, bounding_box_h),
        'rejection_reason': None,
    }

    if candidate_area_pixels < minimum_blob_area_pixels:
        candidate_properties['rejection_reason'] = f'area {candidate_area_pixels:.0f} < {minimum_blob_area_pixels}'
        return None, candidate_properties
    if not (minimum_aspect_ratio_for_cube <= bounding_box_aspect_ratio <= maximum_aspect_ratio_for_cube):
        candidate_properties['rejection_reason'] = (
            f'aspect {bounding_box_aspect_ratio:.2f} outside '
            f'[{minimum_aspect_ratio_for_cube},{maximum_aspect_ratio_for_cube}]')
        return None, candidate_properties
    if candidate_solidity < minimum_solidity_for_cube:
        candidate_properties['rejection_reason'] = f'solidity {candidate_solidity:.2f} < {minimum_solidity_for_cube}'
        return None, candidate_properties

    frame_area_pixels = CAMERA_WIDTH_PIXELS * CAMERA_HEIGHT_PIXELS
    normalized_area_score = min(candidate_area_pixels / (frame_area_pixels * 0.25), 1.0)
    squareness_score = 1.0 - min(abs(math.log(bounding_box_aspect_ratio)) / math.log(2.8), 1.0)

    final_candidate_score = (
        scoring_weight_for_area       * normalized_area_score +
        scoring_weight_for_solidity   * candidate_solidity    +
        scoring_weight_for_squareness * squareness_score
    )
    candidate_properties['score'] = final_candidate_score
    return final_candidate_score, candidate_properties


def detect_red_cube_from_hsv(frame_in_hsv):
    """Run cube detection on a pre-converted HSV image. Returns dict or None."""
    red_mask = build_red_color_mask_from_hsv(frame_in_hsv)
    detected_contours, _ = cv2.findContours(red_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best_score_seen      = -1.0
    best_contour_seen    = None
    best_properties_seen = None

    for candidate_contour in detected_contours:
        candidate_score, candidate_properties = score_cube_candidate_contour(candidate_contour)
        if candidate_score is not None and candidate_score > best_score_seen:
            best_score_seen      = candidate_score
            best_contour_seen    = candidate_contour
            best_properties_seen = candidate_properties

    if best_contour_seen is None:
        return None

    contour_moments = cv2.moments(best_contour_seen)
    if contour_moments['m00'] == 0:
        return None
    centroid_x_pixels = contour_moments['m10'] / contour_moments['m00']
    centroid_y_pixels = contour_moments['m01'] / contour_moments['m00']

    cube_x_normalized = (centroid_x_pixels - CAMERA_WIDTH_PIXELS  / 2) / (CAMERA_WIDTH_PIXELS  / 2)
    cube_y_normalized = (CAMERA_HEIGHT_PIXELS / 2 - centroid_y_pixels) / (CAMERA_HEIGHT_PIXELS / 2)

    cube_distance_meters = distance_calibration_constant / math.sqrt(best_properties_seen['area'])

    return {
        'cube_x_normalized':     cube_x_normalized,
        'cube_y_normalized':     cube_y_normalized,
        'cube_blob_area_pixels': best_properties_seen['area'],
        'cube_distance_meters':  cube_distance_meters,
        'cube_bounding_box':     best_properties_seen['bounding_box'],
        'cube_solidity':         best_properties_seen['solidity'],
        'cube_score':            best_score_seen,
    }


def detect_red_cube(captured_frame_bgr):
    """Wrapper: takes BGR, converts internally. Used by smoke tests and any
    standalone debugging. Combined detection in §6 uses _from_hsv directly."""
    frame_in_hsv = cv2.cvtColor(captured_frame_bgr, cv2.COLOR_BGR2HSV)
    return detect_red_cube_from_hsv(frame_in_hsv)


print("Cube detection functions defined.")


### §4.1 Smoke test for cube detection

Place the cube in front of the bot and run.

In [ ]:
test_frame_bgr = camera_instance.value
test_cube_detection = detect_red_cube(test_frame_bgr)

if test_cube_detection is None:
    print("No red cube detected.")
else:
    print("Cube detection succeeded:")
    for key_name, value in test_cube_detection.items():
        print(f"  {key_name}: {value}")


## §5. Chart detection (NEW)

Same overall pattern as cube detection — build a binary mask, find contours, score them, pick the best — but with two key differences:

1. **Multiple charts of the same color can be visible at once** (two yellow bots in view). Returns a list, not a single best.
2. **Shape classification** (circle vs square) is added on top of color detection. Uses the standard isoperimetric circularity formula `4π × area / perimeter²` — circle ≈ 1.0, square ≈ π/4 ≈ 0.785. We require a non-foreshortened aspect ratio for a confident classification; otherwise report `'unknown'`.

### §5.1 Chart color mask

In [ ]:
def build_chart_color_mask_from_hsv(frame_in_hsv, color_name):
    """Binary mask for the given chart color. Caller has already done cvtColor."""
    if color_name not in chart_color_hsv_ranges:
        raise KeyError(f"Unknown chart color: {color_name!r}; known: {list(chart_color_hsv_ranges)}")

    lower_hsv_threshold, upper_hsv_threshold = chart_color_hsv_ranges[color_name]
    raw_color_mask = cv2.inRange(frame_in_hsv, lower_hsv_threshold, upper_hsv_threshold)

    open_kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                             (chart_morphology_open_kernel_size,
                                              chart_morphology_open_kernel_size))
    close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                             (chart_morphology_close_kernel_size,
                                              chart_morphology_close_kernel_size))
    cleaned_color_mask = cv2.morphologyEx(raw_color_mask,    cv2.MORPH_OPEN,  open_kernel)
    cleaned_color_mask = cv2.morphologyEx(cleaned_color_mask, cv2.MORPH_CLOSE, close_kernel)
    return cleaned_color_mask


print("Chart color mask function defined.")


### §5.2 Shape classification

`circularity = 4π × area / perimeter²`

| shape   | theoretical | typical observed |
|---------|-------------|------------------|
| circle  | 1.000       | 0.92 – 0.98      |
| square  | 0.785       | 0.74 – 0.82      |

Foreshortening (chart face seen at >30° off-axis) collapses both toward 0.83-0.87 — neither value is reliably one shape or the other. We use bbox aspect ratio as the foreshortening detector.

In [ ]:
def classify_chart_shape(candidate_contour, candidate_area_pixels, bbox_aspect_ratio):
    """Return ('circle' | 'square' | 'unknown', circularity_value).

    Returns 'unknown' when the chart is too foreshortened to classify reliably,
    even if circularity happens to land in one of the named bands.
    """
    candidate_perimeter_pixels = cv2.arcLength(candidate_contour, closed=True)
    if candidate_perimeter_pixels < 1.0:
        return 'unknown', 0.0

    circularity_value = 4.0 * math.pi * candidate_area_pixels / (candidate_perimeter_pixels ** 2)

    # Foreshortening check: if aspect is significantly off 1:1, both shapes look similar.
    if abs(math.log(bbox_aspect_ratio)) > maximum_aspect_log_ratio_for_shape_classification:
        return 'unknown', float(circularity_value)

    if circularity_value >= circularity_threshold_for_circle:
        return 'circle', float(circularity_value)
    if circularity_value <= circularity_threshold_for_square:
        return 'square', float(circularity_value)
    return 'unknown', float(circularity_value)


print("Shape classification function defined.")


### §5.3 Chart-candidate scoring

Stricter than cube scoring:
- area weight unchanged
- solidity weight raised (charts are flat printed shapes — should be near-perfectly convex; deviations indicate noise)
- "aspect closeness to 1:1" replaces "squareness" — a clean chart always has aspect near 1.0 regardless of shape

In [ ]:
def score_chart_candidate_contour(candidate_contour):
    candidate_area_pixels = float(cv2.contourArea(candidate_contour))
    bbox_x, bbox_y, bbox_w, bbox_h = cv2.boundingRect(candidate_contour)
    bbox_aspect_ratio = bbox_w / max(bbox_h, 1)

    convex_hull = cv2.convexHull(candidate_contour)
    convex_hull_area = float(cv2.contourArea(convex_hull))
    candidate_solidity = candidate_area_pixels / max(convex_hull_area, 1.0)

    candidate_properties = {
        'area':         candidate_area_pixels,
        'aspect_ratio': bbox_aspect_ratio,
        'solidity':     candidate_solidity,
        'bounding_box': (bbox_x, bbox_y, bbox_w, bbox_h),
        'rejection_reason': None,
    }

    if candidate_area_pixels < minimum_chart_blob_area_pixels:
        candidate_properties['rejection_reason'] = (
            f'area {candidate_area_pixels:.0f} < {minimum_chart_blob_area_pixels}')
        return None, candidate_properties
    if not (minimum_chart_aspect_ratio <= bbox_aspect_ratio <= maximum_chart_aspect_ratio):
        candidate_properties['rejection_reason'] = (
            f'aspect {bbox_aspect_ratio:.2f} outside '
            f'[{minimum_chart_aspect_ratio},{maximum_chart_aspect_ratio}]')
        return None, candidate_properties
    if candidate_solidity < minimum_chart_solidity:
        candidate_properties['rejection_reason'] = (
            f'solidity {candidate_solidity:.2f} < {minimum_chart_solidity}')
        return None, candidate_properties

    # Score components
    frame_area_pixels = CAMERA_WIDTH_PIXELS * CAMERA_HEIGHT_PIXELS
    normalized_area_score = min(candidate_area_pixels / (frame_area_pixels * 0.10), 1.0)
    aspect_closeness_score = 1.0 - min(abs(math.log(bbox_aspect_ratio)) / math.log(2.0), 1.0)

    final_score = (
        chart_scoring_weight_for_area              * normalized_area_score +
        chart_scoring_weight_for_solidity          * candidate_solidity    +
        chart_scoring_weight_for_aspect_closeness  * aspect_closeness_score
    )
    candidate_properties['score'] = final_score
    return final_score, candidate_properties


print("Chart scoring function defined.")


### §5.4 Per-frame chart detection

Returns **all** charts of the given color that pass the acceptance filters, sorted by score. We don't cap the list because the score-based filter is already strict — pathological multi-detection cases (>2 charts of one color) indicate something else is wrong and should surface in the data, not be hidden.

In [ ]:
def detect_charts_of_color_from_hsv(frame_in_hsv, color_name):
    """Return a list of chart-detection dicts for all charts of the given color.

    Each detection has the shape committed in the design doc:
      peer_color, peer_shape, peer_x_normalized, peer_y_normalized,
      peer_distance_meters, peer_blob_area_pixels, peer_shape_confidence,
      peer_bounding_box, peer_score
    """
    color_mask = build_chart_color_mask_from_hsv(frame_in_hsv, color_name)
    detected_contours, _ = cv2.findContours(color_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    chart_detections = []
    for candidate_contour in detected_contours:
        candidate_score, candidate_properties = score_chart_candidate_contour(candidate_contour)
        if candidate_score is None:
            continue

        # Centroid in normalized coordinates
        contour_moments = cv2.moments(candidate_contour)
        if contour_moments['m00'] == 0:
            continue
        centroid_x_pixels = contour_moments['m10'] / contour_moments['m00']
        centroid_y_pixels = contour_moments['m01'] / contour_moments['m00']
        peer_x_normalized = (centroid_x_pixels - CAMERA_WIDTH_PIXELS  / 2) / (CAMERA_WIDTH_PIXELS  / 2)
        peer_y_normalized = (CAMERA_HEIGHT_PIXELS / 2 - centroid_y_pixels) / (CAMERA_HEIGHT_PIXELS / 2)

        # Shape classification
        chart_shape_name, chart_shape_circularity = classify_chart_shape(
            candidate_contour,
            candidate_properties['area'],
            candidate_properties['aspect_ratio'],
        )

        # Distance using per-(color, shape) k
        chart_k_constant = get_chart_distance_constant(color_name, chart_shape_name)
        if chart_k_constant > 0 and candidate_properties['area'] > 0:
            peer_distance_meters = chart_k_constant / math.sqrt(candidate_properties['area'])
        else:
            peer_distance_meters = -1.0  # uncalibrated or invalid

        chart_detections.append({
            'peer_color':            color_name,
            'peer_shape':            chart_shape_name,
            'peer_x_normalized':     float(peer_x_normalized),
            'peer_y_normalized':     float(peer_y_normalized),
            'peer_distance_meters':  float(peer_distance_meters),
            'peer_blob_area_pixels': float(candidate_properties['area']),
            'peer_shape_confidence': float(chart_shape_circularity),
            'peer_bounding_box':     [int(v) for v in candidate_properties['bounding_box']],
            'peer_score':            float(candidate_score),
        })

    # Best-scored detections first.
    chart_detections.sort(key=lambda detection: detection['peer_score'], reverse=True)
    return chart_detections


print("Per-color chart detection function defined.")


## §6. Combined cube + peers detection

This is the function the detection thread actually calls. **One BGR→HSV conversion per frame**, shared between cube detection and all peer-color detections. With 2 chart colors, that means 1 cvtColor + 3 inRange operations instead of 4 cvtColor + 4 inRange — the conversion alone is ~5 ms at 720×720, so this saves ~15 ms per frame on the Nano.

### §6.1 `detect_cube_and_peers`

In [ ]:
def detect_cube_and_peers(captured_frame_bgr):
    """Run cube + all peer-color detections from a single BGR→HSV conversion.

    Returns (cube_detection_or_None, peer_detections_list).

    If identity is assigned, the bot's own (color, shape) is filtered out of
    the peer list. Until identity is assigned (None, None), all detections
    pass through — fail-open.
    """
    frame_in_hsv = cv2.cvtColor(captured_frame_bgr, cv2.COLOR_BGR2HSV)

    cube_detection = detect_red_cube_from_hsv(frame_in_hsv)

    my_assigned_color, my_assigned_shape = get_assigned_identity()
    is_identity_set = (my_assigned_color is not None) and (my_assigned_shape is not None)

    peer_detections = []
    for color_name in chart_color_hsv_ranges.keys():
        for chart_detection in detect_charts_of_color_from_hsv(frame_in_hsv, color_name):
            if (is_identity_set
                    and chart_detection['peer_color'] == my_assigned_color
                    and chart_detection['peer_shape'] == my_assigned_shape):
                # This is presumably this bot's own chart reflected somewhere
                # (mirror, glass) or a misidentification. Drop it.
                continue
            peer_detections.append(chart_detection)

    return cube_detection, peer_detections


print("Combined cube + peers detection function defined.")


### §6.2 Smoke test for combined detection

In [ ]:
test_frame_bgr = camera_instance.value
test_cube_detection, test_peer_detections = detect_cube_and_peers(test_frame_bgr)

print("=== Combined detection result ===")
if test_cube_detection is None:
    print("Cube: NOT detected.")
else:
    print(f"Cube: detected at x={test_cube_detection['cube_x_normalized']:+.3f} "
          f"y={test_cube_detection['cube_y_normalized']:+.3f} "
          f"distance={test_cube_detection['cube_distance_meters']:.2f} m "
          f"area={test_cube_detection['cube_blob_area_pixels']:.0f} px")

print(f"Peers: {len(test_peer_detections)} detected.")
for peer_index, peer in enumerate(test_peer_detections):
    print(f"  [{peer_index}] {peer['peer_color']:6s}/{peer['peer_shape']:7s} "
          f"x={peer['peer_x_normalized']:+.3f} y={peer['peer_y_normalized']:+.3f} "
          f"distance={peer['peer_distance_meters']:.2f} m "
          f"area={peer['peer_blob_area_pixels']:.0f} px "
          f"circularity={peer['peer_shape_confidence']:.3f} "
          f"score={peer['peer_score']:.3f}")

assigned_now = get_assigned_identity()
print(f"\nCurrent identity: color={assigned_now[0]!r} shape={assigned_now[1]!r}")
if assigned_now == (None, None):
    print("(Identity not yet assigned — own chart, if any, will appear in peer list.)")


## §7. Distance calibration

### §7.1 Cube distance calibration (carried over from Phase 1b)

In [ ]:
def calibrate_distance_constant(known_distance_meters, samples_to_average=15):
    global distance_calibration_constant

    collected_blob_areas = []
    for sample_index in range(samples_to_average):
        captured_frame_bgr = camera_instance.value
        detection_result = detect_red_cube(captured_frame_bgr)
        if detection_result is not None:
            collected_blob_areas.append(detection_result['cube_blob_area_pixels'])
        time.sleep(0.05)

    if len(collected_blob_areas) < samples_to_average // 2:
        print(f"Calibration failed: only {len(collected_blob_areas)} valid samples.")
        return None

    median_blob_area = float(np.median(collected_blob_areas))
    new_calibration_constant = known_distance_meters * math.sqrt(median_blob_area)
    distance_calibration_constant = new_calibration_constant

    print(f"Median blob area at {known_distance_meters} m: {median_blob_area:.1f} px")
    print(f"Updated distance_calibration_constant = {new_calibration_constant:.2f}")
    return new_calibration_constant


print("Cube calibration function ready.")
print("Place cube at known distance and run, e.g.:")
print("  calibrate_distance_constant(known_distance_meters=1.0)")


### §7.2 Per-(color, shape) chart distance calibration

Run **once per (color, shape) combination** — four runs total. Place the named chart at the known distance, with the chart facing the camera squarely. Off-axis calibration will bias k low.

In [ ]:
def calibrate_chart_distance_constant(color_name, shape_name,
                                       known_distance_meters,
                                       samples_to_average=15):
    """Calibrate k for one (color, shape) pair. Updates the in-memory dict."""
    if color_name not in valid_chart_colors:
        print(f"Unknown color {color_name!r}; valid: {sorted(valid_chart_colors)}")
        return None
    if shape_name not in valid_chart_shapes:
        print(f"Unknown shape {shape_name!r}; valid: {sorted(valid_chart_shapes)}")
        return None

    collected_blob_areas = []
    classifications_seen = []
    for sample_index in range(samples_to_average):
        captured_frame_bgr = camera_instance.value
        frame_in_hsv = cv2.cvtColor(captured_frame_bgr, cv2.COLOR_BGR2HSV)
        chart_detections = detect_charts_of_color_from_hsv(frame_in_hsv, color_name)
        if chart_detections:
            best_detection = chart_detections[0]  # already sorted by score
            collected_blob_areas.append(best_detection['peer_blob_area_pixels'])
            classifications_seen.append(best_detection['peer_shape'])
        time.sleep(0.05)

    if len(collected_blob_areas) < samples_to_average // 2:
        print(f"Calibration failed: only {len(collected_blob_areas)} valid samples for {color_name}.")
        print("Check chart is visible, in frame, and HSV thresholds for this color are correct.")
        return None

    # Sanity check: did we mostly classify it as the expected shape?
    correct_classifications = sum(1 for s in classifications_seen if s == shape_name)
    classification_fraction = correct_classifications / len(classifications_seen)
    print(f"Shape classification: {correct_classifications}/{len(classifications_seen)} "
          f"frames classified as {shape_name!r} ({classification_fraction*100:.0f}%)")
    if classification_fraction < 0.5:
        print(f"  WARNING: most frames were classified as something else. "
              f"Consider adjusting circularity thresholds or recheck the chart.")

    median_blob_area = float(np.median(collected_blob_areas))
    new_chart_k = known_distance_meters * math.sqrt(median_blob_area)
    chart_distance_calibration_constants[(color_name, shape_name)] = new_chart_k

    print(f"Median chart area at {known_distance_meters} m: {median_blob_area:.1f} px")
    print(f"Updated chart_distance_calibration_constants[({color_name!r}, {shape_name!r})] = {new_chart_k:.2f}")
    return new_chart_k


print("Chart calibration function ready. Run for each (color, shape):")
print("  calibrate_chart_distance_constant('yellow', 'circle', known_distance_meters=1.0)")
print("  calibrate_chart_distance_constant('yellow', 'square', known_distance_meters=1.0)")
print("  calibrate_chart_distance_constant('blue',   'circle', known_distance_meters=1.0)")
print("  calibrate_chart_distance_constant('blue',   'square', known_distance_meters=1.0)")


## §8. In-situ HSV recalibration

Run only if the default HSV ranges aren't catching the cube or a specific chart in this room's lighting.

### §8.1 Red recalibration (carried over from Phase 1b)

In [ ]:
def recalibrate_red_hsv_thresholds(samples_to_collect=20, hue_margin=8,
                                    sat_margin=40, val_margin=40):
    global red_hue_lower_range_min, red_hue_lower_range_max
    global red_hue_upper_range_min, red_hue_upper_range_max

    collected_pixel_hsv_values = []
    for sample_index in range(samples_to_collect):
        captured_frame_bgr = camera_instance.value
        detection_result = detect_red_cube(captured_frame_bgr)
        if detection_result is None:
            time.sleep(0.05)
            continue
        bx, by, bw, bh = detection_result['cube_bounding_box']
        frame_in_hsv = cv2.cvtColor(captured_frame_bgr, cv2.COLOR_BGR2HSV)
        cube_region_hsv = frame_in_hsv[by:by+bh, bx:bx+bw]
        cube_region_mask = build_red_color_mask_from_hsv(frame_in_hsv)[by:by+bh, bx:bx+bw]
        red_pixels_in_region = cube_region_hsv[cube_region_mask > 0]
        if len(red_pixels_in_region) > 0:
            collected_pixel_hsv_values.append(red_pixels_in_region)
        time.sleep(0.05)

    if len(collected_pixel_hsv_values) == 0:
        print("Recalibration failed: cube not detected with current thresholds.")
        return

    all_red_pixels = np.vstack(collected_pixel_hsv_values)
    print(f"Collected {len(all_red_pixels)} red pixels across {len(collected_pixel_hsv_values)} frames.")

    is_low_hue_pixel  = all_red_pixels[:, 0] < 90
    low_hue_pixels  = all_red_pixels[is_low_hue_pixel]
    high_hue_pixels = all_red_pixels[~is_low_hue_pixel]

    def compute_range_from_pixels(pixel_array, name_for_logging):
        if len(pixel_array) < 50:
            return None
        h_low,  h_high = np.percentile(pixel_array[:, 0], [ 5, 95])
        s_low,  s_high = np.percentile(pixel_array[:, 1], [ 5, 95])
        v_low,  v_high = np.percentile(pixel_array[:, 2], [ 5, 95])
        print(f"  {name_for_logging}: H[{h_low:.0f}-{h_high:.0f}] "
              f"S[{s_low:.0f}-{s_high:.0f}] V[{v_low:.0f}-{v_high:.0f}]")
        return (
            np.array([max(  0, h_low  - hue_margin),
                      max(  0, s_low  - sat_margin),
                      max(  0, v_low  - val_margin)], dtype=np.uint8),
            np.array([min(180, h_high + hue_margin),
                      min(255, s_high + sat_margin),
                      min(255, v_high + val_margin)], dtype=np.uint8),
        )

    new_lower_range = compute_range_from_pixels(low_hue_pixels,  "low-hue red")
    new_upper_range = compute_range_from_pixels(high_hue_pixels, "high-hue red")

    if new_lower_range is not None:
        red_hue_lower_range_min, red_hue_lower_range_max = new_lower_range
    if new_upper_range is not None:
        red_hue_upper_range_min, red_hue_upper_range_max = new_upper_range

    print("Updated thresholds:")
    print(f"  lower: {red_hue_lower_range_min} -> {red_hue_lower_range_max}")
    print(f"  upper: {red_hue_upper_range_min} -> {red_hue_upper_range_max}")


print("Red HSV recalibration function ready.")


### §8.2 Per-color chart HSV recalibration

Place the chart of the named color in front of the bot, run. Function uses the **best-scoring** detection of that color in each frame to derive the actual hue/sat/val distribution, expands by margins, and updates `chart_color_hsv_ranges`.

In [ ]:
def recalibrate_chart_hsv_thresholds(color_name, samples_to_collect=20,
                                       hue_margin=6, sat_margin=40, val_margin=40):
    """Recalibrate HSV range for one chart color. Updates chart_color_hsv_ranges."""
    if color_name not in chart_color_hsv_ranges:
        print(f"Unknown chart color {color_name!r}; valid: {sorted(chart_color_hsv_ranges)}")
        return

    collected_pixel_hsv_values = []
    for sample_index in range(samples_to_collect):
        captured_frame_bgr = camera_instance.value
        frame_in_hsv = cv2.cvtColor(captured_frame_bgr, cv2.COLOR_BGR2HSV)
        chart_detections = detect_charts_of_color_from_hsv(frame_in_hsv, color_name)
        if not chart_detections:
            time.sleep(0.05)
            continue
        best_detection = chart_detections[0]
        bx, by, bw, bh = best_detection['peer_bounding_box']
        chart_region_hsv  = frame_in_hsv[by:by+bh, bx:bx+bw]
        chart_region_mask = build_chart_color_mask_from_hsv(frame_in_hsv, color_name)[by:by+bh, bx:bx+bw]
        pixels_in_region = chart_region_hsv[chart_region_mask > 0]
        if len(pixels_in_region) > 0:
            collected_pixel_hsv_values.append(pixels_in_region)
        time.sleep(0.05)

    if len(collected_pixel_hsv_values) == 0:
        print(f"Recalibration failed: chart of color {color_name!r} not detected with current thresholds.")
        return

    all_chart_pixels = np.vstack(collected_pixel_hsv_values)
    print(f"Collected {len(all_chart_pixels)} {color_name} pixels "
          f"across {len(collected_pixel_hsv_values)} frames.")

    h_low,  h_high = np.percentile(all_chart_pixels[:, 0], [ 5, 95])
    s_low,  s_high = np.percentile(all_chart_pixels[:, 1], [ 5, 95])
    v_low,  v_high = np.percentile(all_chart_pixels[:, 2], [ 5, 95])
    print(f"  observed: H[{h_low:.0f}-{h_high:.0f}] "
          f"S[{s_low:.0f}-{s_high:.0f}] V[{v_low:.0f}-{v_high:.0f}]")

    new_lower = np.array([max(  0, h_low  - hue_margin),
                          max(  0, s_low  - sat_margin),
                          max(  0, v_low  - val_margin)], dtype=np.uint8)
    new_upper = np.array([min(180, h_high + hue_margin),
                          min(255, s_high + sat_margin),
                          min(255, v_high + val_margin)], dtype=np.uint8)

    chart_color_hsv_ranges[color_name] = (new_lower, new_upper)
    print(f"Updated chart_color_hsv_ranges[{color_name!r}] = "
          f"({new_lower.tolist()} -> {new_upper.tolist()})")


print("Chart HSV recalibration function ready.")
print("Run as: recalibrate_chart_hsv_thresholds('yellow')  or  ('blue')")


## §9. Background detection thread

Refreshes `latest_sensor_state` at `1 / detection_loop_period_seconds` Hz. Calls `detect_cube_and_peers` (one BGR→HSV pass) and the same `compute_frame_difference` from Phase 1b.

The HTTP handlers in §10 read `latest_sensor_state` under the lock and return immediately — no detection cost on the network path.

In [ ]:
sensor_state_lock = threading.Lock()
latest_detection_bounding_box = None  # cube bbox for /camera_annotated; held under sensor_state_lock

latest_sensor_state = {
    'timestamp_seconds':         0.0,
    'cube_visible':              False,
    'cube_x_normalized':         0.0,
    'cube_y_normalized':         0.0,
    'cube_distance_meters':      -1.0,
    'frame_difference_value':    0.0,
    'peers':                     [],
    'detection_loop_iteration':  0,
}

detection_thread_should_run = False


def compute_frame_difference(current_frame_bgr, previous_frame_bgr):
    """Mean absolute pixel difference between two frames, downsampled to 32x32."""
    if previous_frame_bgr is None:
        return 0.0
    small_current_frame  = cv2.resize(current_frame_bgr,  (32, 32))
    small_previous_frame = cv2.resize(previous_frame_bgr, (32, 32))
    return float(np.mean(cv2.absdiff(small_current_frame, small_previous_frame)))


def run_detection_loop():
    """Background loop: capture, detect cube + peers, update shared state."""
    global latest_sensor_state, latest_detection_bounding_box
    previous_frame_bgr = None
    detection_loop_iteration_counter = 0

    while detection_thread_should_run:
        loop_iteration_start_time = time.time()

        captured_frame_bgr = camera_instance.value
        cube_detection, peer_detections = detect_cube_and_peers(captured_frame_bgr)
        frame_difference_value = compute_frame_difference(captured_frame_bgr, previous_frame_bgr)
        previous_frame_bgr = captured_frame_bgr.copy()

        detection_loop_iteration_counter += 1

        if cube_detection is None:
            new_latest_detection_bounding_box = None
            new_sensor_state = {
                'timestamp_seconds':        time.time(),
                'cube_visible':             False,
                'cube_x_normalized':        0.0,
                'cube_y_normalized':        0.0,
                'cube_distance_meters':     -1.0,
                'frame_difference_value':   frame_difference_value,
                'peers':                    peer_detections,
                'detection_loop_iteration': detection_loop_iteration_counter,
            }
        else:
            new_latest_detection_bounding_box = cube_detection['cube_bounding_box']
            new_sensor_state = {
                'timestamp_seconds':        time.time(),
                'cube_visible':             True,
                'cube_x_normalized':        cube_detection['cube_x_normalized'],
                'cube_y_normalized':        cube_detection['cube_y_normalized'],
                'cube_distance_meters':     cube_detection['cube_distance_meters'],
                'cube_bounding_box':        [int(v) for v in cube_detection['cube_bounding_box']],
                'frame_difference_value':   frame_difference_value,
                'peers':                    peer_detections,
                'detection_loop_iteration': detection_loop_iteration_counter,
            }

        with sensor_state_lock:
            latest_sensor_state = new_sensor_state
            latest_detection_bounding_box = new_latest_detection_bounding_box

        time_elapsed = time.time() - loop_iteration_start_time
        time_to_sleep = detection_loop_period_seconds - time_elapsed
        if time_to_sleep > 0:
            time.sleep(time_to_sleep)


def start_detection_thread():
    global detection_thread_should_run
    if detection_thread_should_run:
        print("Detection thread already running.")
        return
    detection_thread_should_run = True
    background_detection_thread = threading.Thread(target=run_detection_loop, daemon=True)
    background_detection_thread.start()
    print("Detection thread started.")


def stop_detection_thread():
    global detection_thread_should_run
    detection_thread_should_run = False
    print("Detection thread stop requested.")


print("Detection thread helpers defined.")


## §10. HTTP server

Endpoints:
- `GET  /state`           — JSON of `latest_sensor_state` + assigned identity
- `POST /command`         — body `{left_motor_speed, right_motor_speed, command_ttl_milliseconds}`
- `POST /assign_identity` — body `{assigned_color, assigned_shape}` — sets identity; validates against `valid_chart_colors` and `valid_chart_shapes`
- `GET  /camera`          — raw JPEG
- `GET  /camera_annotated`— JPEG with cube bbox (green) and peer bboxes (color-coded) + labels
- `GET  /health`          — JSON liveness + assigned identity

The watchdog is unchanged from Phase 1b: every accepted `/command` (re)arms a timer; if no new command arrives within `command_ttl_milliseconds`, motors stop.

In [ ]:
# --- Command watchdog state ---
command_watchdog_lock = threading.Lock()
active_command_watchdog_timer = None


def apply_motor_command(left_motor_speed, right_motor_speed, command_ttl_milliseconds):
    global active_command_watchdog_timer
    robot_instance.set_motors(float(left_motor_speed), float(right_motor_speed))

    with command_watchdog_lock:
        if active_command_watchdog_timer is not None:
            active_command_watchdog_timer.cancel()

        watchdog_timeout_seconds = max(0.05, command_ttl_milliseconds / 1000.0)
        new_watchdog_timer = threading.Timer(
            watchdog_timeout_seconds,
            lambda: robot_instance.stop(),
        )
        new_watchdog_timer.daemon = True
        new_watchdog_timer.start()
        active_command_watchdog_timer = new_watchdog_timer


def cancel_command_watchdog_and_stop():
    global active_command_watchdog_timer
    with command_watchdog_lock:
        if active_command_watchdog_timer is not None:
            active_command_watchdog_timer.cancel()
            active_command_watchdog_timer = None
    robot_instance.stop()


def bgr8_to_jpeg(frame_bgr):
    return cv2.imencode('.jpg', frame_bgr)[1].tobytes()


# Per-color BGR for drawing peer bboxes on /camera_annotated
peer_bbox_color_for_color_name = {
    'yellow': (  0, 255, 255),  # BGR yellow
    'blue':   (255, 100,   0),  # BGR blue
}


def annotate_frame_for_debug_view(captured_frame_bgr, current_cube_bounding_box,
                                   current_peer_detections):
    """Draw center reference line, cube bbox, peer bboxes + labels."""
    annotated_frame_bgr = captured_frame_bgr.copy()
    frame_height_pixels, frame_width_pixels = annotated_frame_bgr.shape[:2]

    # Center reference line (yellow, 1px)
    frame_center_x_pixels = frame_width_pixels // 2
    cv2.line(annotated_frame_bgr,
             (frame_center_x_pixels, 0),
             (frame_center_x_pixels, frame_height_pixels),
             (0, 255, 255), 1)

    # Cube bbox (green, 2px)
    if current_cube_bounding_box is not None:
        cx, cy, cw, ch = current_cube_bounding_box
        cv2.rectangle(annotated_frame_bgr,
                      (cx, cy), (cx + cw, cy + ch),
                      (0, 255, 0), 2)
        cv2.putText(annotated_frame_bgr, 'cube', (cx, max(0, cy - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    # Peer bboxes
    for peer in current_peer_detections:
        peer_bbox = peer.get('peer_bounding_box')
        if peer_bbox is None or len(peer_bbox) != 4:
            continue
        px, py, pw, ph = peer_bbox
        bbox_color = peer_bbox_color_for_color_name.get(peer['peer_color'], (200, 200, 200))
        cv2.rectangle(annotated_frame_bgr,
                      (px, py), (px + pw, py + ph),
                      bbox_color, 2)
        peer_label = f"{peer['peer_color']}/{peer['peer_shape']}"
        cv2.putText(annotated_frame_bgr, peer_label, (px, max(0, py - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, bbox_color, 1)

    return annotated_frame_bgr


class JetbotSensorRequestHandler(BaseHTTPRequestHandler):
    def log_message(self, format_string, *args):
        return  # suppress per-request stdout spam

    def _respond_with_json(self, http_status_code, payload_dict):
        encoded_response_body = json_module.dumps(payload_dict).encode('utf-8')
        self.send_response(http_status_code)
        self.send_header('Content-Type',   'application/json')
        self.send_header('Content-Length', str(len(encoded_response_body)))
        self.end_headers()
        self.wfile.write(encoded_response_body)

    def _respond_with_text(self, http_status_code, message_text):
        encoded = message_text.encode('utf-8')
        self.send_response(http_status_code)
        self.send_header('Content-Type',   'text/plain')
        self.send_header('Content-Length', str(len(encoded)))
        self.end_headers()
        self.wfile.write(encoded)

    def do_GET(self):
        parsed_url_path = urllib.parse.urlparse(self.path).path

        if parsed_url_path == '/state':
            with sensor_state_lock:
                snapshot_of_sensor_state = dict(latest_sensor_state)
            current_assigned_color, current_assigned_shape = get_assigned_identity()
            snapshot_of_sensor_state['assigned_color'] = current_assigned_color
            snapshot_of_sensor_state['assigned_shape'] = current_assigned_shape
            self._respond_with_json(200, snapshot_of_sensor_state)

        elif parsed_url_path == '/camera':
            captured_frame_bgr = camera_instance.value
            jpeg_bytes = bgr8_to_jpeg(captured_frame_bgr)
            self.send_response(200)
            self.send_header('Content-Type',   'image/jpeg')
            self.send_header('Content-Length', str(len(jpeg_bytes)))
            self.end_headers()
            self.wfile.write(jpeg_bytes)

        elif parsed_url_path == '/camera_annotated':
            captured_frame_bgr = camera_instance.value
            with sensor_state_lock:
                snapshot_cube_bbox = latest_detection_bounding_box
                # Shallow copy of peers list; entries are dicts not mutated by reader.
                snapshot_peers = list(latest_sensor_state.get('peers', []))
            annotated_frame_bgr = annotate_frame_for_debug_view(
                captured_frame_bgr, snapshot_cube_bbox, snapshot_peers)
            jpeg_bytes = bgr8_to_jpeg(annotated_frame_bgr)
            self.send_response(200)
            self.send_header('Content-Type',   'image/jpeg')
            self.send_header('Content-Length', str(len(jpeg_bytes)))
            self.end_headers()
            self.wfile.write(jpeg_bytes)

        elif parsed_url_path == '/health':
            current_assigned_color, current_assigned_shape = get_assigned_identity()
            self._respond_with_json(200, {
                'status':                    'ok',
                'detection_thread_running':  detection_thread_should_run,
                'detection_loop_iteration':  latest_sensor_state.get('detection_loop_iteration', 0),
                'assigned_color':            current_assigned_color,
                'assigned_shape':            current_assigned_shape,
            })

        else:
            self._respond_with_text(404, f"Unknown path: {parsed_url_path}")

    def do_POST(self):
        parsed_url_path = urllib.parse.urlparse(self.path).path

        if parsed_url_path == '/command':
            try:
                request_body_length = int(self.headers.get('Content-Length', '0'))
                raw_request_body    = self.rfile.read(request_body_length)
                parsed_command_body = json_module.loads(raw_request_body.decode('utf-8'))

                left_motor_speed         = float(parsed_command_body['left_motor_speed'])
                right_motor_speed        = float(parsed_command_body['right_motor_speed'])
                command_ttl_milliseconds = int(parsed_command_body.get(
                    'command_ttl_milliseconds', default_command_ttl_milliseconds))

                left_motor_speed  = max(-1.0, min(1.0, left_motor_speed))
                right_motor_speed = max(-1.0, min(1.0, right_motor_speed))

                apply_motor_command(left_motor_speed, right_motor_speed, command_ttl_milliseconds)
                self._respond_with_json(200, {
                    'accepted_left_motor_speed':  left_motor_speed,
                    'accepted_right_motor_speed': right_motor_speed,
                    'watchdog_ttl_milliseconds':  command_ttl_milliseconds,
                })

            except (ValueError, KeyError, json_module.JSONDecodeError) as parse_error:
                self._respond_with_json(400, {'error': f'bad command body: {parse_error!r}'})

        elif parsed_url_path == '/assign_identity':
            try:
                request_body_length = int(self.headers.get('Content-Length', '0'))
                raw_request_body    = self.rfile.read(request_body_length)
                parsed_body         = json_module.loads(raw_request_body.decode('utf-8'))

                requested_color = parsed_body.get('assigned_color')
                requested_shape = parsed_body.get('assigned_shape')

                if requested_color not in valid_chart_colors:
                    self._respond_with_json(400, {
                        'error': f'unknown color: {requested_color!r}',
                        'valid_colors': sorted(valid_chart_colors),
                    })
                    return
                if requested_shape not in valid_chart_shapes:
                    self._respond_with_json(400, {
                        'error': f'unknown shape: {requested_shape!r}',
                        'valid_shapes': sorted(valid_chart_shapes),
                    })
                    return

                set_assigned_identity(requested_color, requested_shape)

                self._respond_with_json(200, {
                    'accepted_color': requested_color,
                    'accepted_shape': requested_shape,
                })

            except (ValueError, KeyError, json_module.JSONDecodeError) as parse_error:
                self._respond_with_json(400, {'error': f'bad request body: {parse_error!r}'})

        else:
            self._respond_with_text(404, f"Unknown path: {parsed_url_path}")


http_server_instance = None
http_server_thread   = None


def start_http_server():
    global http_server_instance, http_server_thread
    if http_server_instance is not None:
        print("HTTP server already running.")
        return
    http_server_instance = HTTPServer(('0.0.0.0', http_server_port), JetbotSensorRequestHandler)
    http_server_thread = threading.Thread(target=http_server_instance.serve_forever, daemon=True)
    http_server_thread.start()
    print(f"HTTP server listening on port {http_server_port}.")


def stop_http_server():
    global http_server_instance, http_server_thread
    if http_server_instance is None:
        print("HTTP server not running.")
        return
    http_server_instance.shutdown()
    http_server_instance.server_close()
    http_server_instance = None
    http_server_thread   = None
    print("HTTP server stopped.")


print("HTTP server helpers defined.")


## §11. Start the server

Run this cell to start serving. The bot will then accept `/state`, `/command`, `/assign_identity`, etc., from any client on the network. Look up the bot's IP with `hostname -I` in a terminal on the bot if you don't already know it.

In [ ]:
start_detection_thread()
start_http_server()

# Local self-test
import urllib.request
try:
    self_test_response = urllib.request.urlopen(
        f'http://127.0.0.1:{http_server_port}/state', timeout=1.0)
    print("Self-test /state response:")
    print(self_test_response.read().decode('utf-8'))
except Exception as self_test_error:
    print(f"Self-test failed: {self_test_error!r}")


## §12. Detection-rate diagnostic (NEW)

Phase 2b adds 2 extra HSV mask + contour passes per frame (one per peer color). On a 720×720 frame on the Jetson Nano CPU (`cv2.*` is CPU code; the Maxwell GPU is not used unless you build OpenCV with CUDA — you don't), this could push the detection thread below the 10 Hz target.

This cell measures actual Hz over a 10-second window. Run it with a real cube and at least one peer chart in view (so the detector is doing real work, not skipping out early on empty masks).

**Decision rules:**
- ≥ 9 Hz: healthy
- 7–9 Hz: acceptable, but client polling should not be set above the actual rate
- < 7 Hz: optimize before continuing to Phase 2c (suggestions printed below)

In [ ]:
def measure_detection_thread_rate(measurement_duration_seconds=10.0):
    state_url = f'http://127.0.0.1:{http_server_port}/state'

    response_at_start = urllib.request.urlopen(state_url, timeout=1.0)
    state_at_start    = json_module.loads(response_at_start.read().decode('utf-8'))
    iteration_at_start = state_at_start['detection_loop_iteration']
    timestamp_at_start = time.time()
    peers_at_start     = len(state_at_start.get('peers', []))
    cube_visible_at_start = state_at_start.get('cube_visible', False)

    print(f"At start: iteration={iteration_at_start}, "
          f"cube_visible={cube_visible_at_start}, peers={peers_at_start}")
    print(f"Measuring for {measurement_duration_seconds:.1f} s...")
    time.sleep(measurement_duration_seconds)

    response_at_end = urllib.request.urlopen(state_url, timeout=1.0)
    state_at_end    = json_module.loads(response_at_end.read().decode('utf-8'))
    iteration_at_end = state_at_end['detection_loop_iteration']
    timestamp_at_end = time.time()
    peers_at_end     = len(state_at_end.get('peers', []))
    cube_visible_at_end = state_at_end.get('cube_visible', False)

    iterations_observed = iteration_at_end - iteration_at_start
    actual_duration_seconds = timestamp_at_end - timestamp_at_start
    measured_hz = iterations_observed / actual_duration_seconds
    target_hz = 1.0 / detection_loop_period_seconds

    print(f"At end:   iteration={iteration_at_end}, "
          f"cube_visible={cube_visible_at_end}, peers={peers_at_end}")
    print(f"Iterations: {iterations_observed} new in {actual_duration_seconds:.2f} s")
    print(f"Measured detection rate: {measured_hz:.2f} Hz  (target: {target_hz:.2f} Hz)")

    if measured_hz < 7.0:
        print()
        print("BELOW 7 Hz — peer detection is too expensive on this hardware.")
        print("Suggested fixes (in order of cost):")
        print("  1. Skip peer detection on alternate frames (cube every iter, peers every other).")
        print("  2. Run peer detection at half resolution: downsample HSV to 360x360 before "
              "build_chart_color_mask_from_hsv, then scale bbox/centroid back up.")
        print("  3. Drop camera resolution to 480x480 (re-calibrate distance constants).")
    elif measured_hz < target_hz * 0.85:
        print(f"Below target. Detection thread is taking >{detection_loop_period_seconds*1000:.0f} ms per iteration.")
        print("Functional, but client polling should not exceed measured rate.")
    else:
        print("Detection rate is healthy.")

    return measured_hz


# Run after the server is up and the detection thread is warm. Make sure cube
# and at least one peer chart are in the camera's view for a realistic measurement.
# measure_detection_thread_rate(measurement_duration_seconds=10.0)
print("Diagnostic ready. Call measure_detection_thread_rate() when ready.")


## §13. Shutdown

In [ ]:
cancel_command_watchdog_and_stop()
stop_detection_thread()
stop_http_server()
robot_instance.stop()
camera_instance.stop()
print("Bot-side shutdown complete.")
